In [10]:
from dotenv import load_dotenv
import os
from pathlib import Path
workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

Loaded environment from: c:\Users\Girish Kulkarni\Downloads\LangChainTrainings\.env
LangSmith tracing enabled for project: LangChainTrainings-Agents


In [11]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
)

In [12]:
# Connect DeepEval to Confident AI
# Put your key in the project-root .env as: CONFIDENT_API_KEY=confident_xxx

import os

from dotenv import load_dotenv

# override=True so edits to .env take effect without restarting the kernel
load_dotenv(override=True)

# qwen3:8b is a reasoning model and can exceed DeepEval's default 88.5s per-attempt timeout
os.environ.setdefault("DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE", "600")

import deepeval
from deepeval.models import OllamaModel

api_key = os.getenv("CONFIDENT_API_KEY")
if not api_key or "paste_your_key" in api_key:
    raise ValueError(
        "Set CONFIDENT_API_KEY in the project-root .env to your real Confident AI key."
    )
deepeval.login(api_key=api_key)

judge = OllamaModel(
    model="qwen3:8b",
    base_url="http://localhost:11434",
    temperature=0.0,
)


🎉🥳 Congratulations! You've successfully logged in! 🙌

In [13]:
# Simple RAG evaluation with DeepEval

from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
from deepeval.test_case import LLMTestCase

question = "What is the capital of France and what is it known for?"

retrieval_context = [
    "France is a country in Western Europe.",
    "Paris is the capital and largest city of France.",
    "Paris is known for the Eiffel Tower and the Louvre Museum.",
]

actual_output = llm.invoke(question).content

test_case = LLMTestCase(
    input=question,
    actual_output=actual_output,
    retrieval_context=retrieval_context,
)

metrics = [
    AnswerRelevancyMetric(model=judge, threshold=0.5),
    FaithfulnessMetric(model=judge, threshold=0.5),
]

result = evaluate(test_cases=[test_case], metrics=metrics)


✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen3:8b (Ollama), strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen3:8b (Ollama), strict=False, 
async_mode=True)...

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

c:\Users\Girish Kulkarni\Downloads\LangChainTrainings\myenv321\Lib\site-packages\rich\live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

KeyboardInterrupt: 

In get_cached_test_run, temp=False, Lock acquisition failed: Shared locks on Windows require the win32 extra 
(pywin32); msvcrt provides no true shared lock. Install it with: pip install "portalocker[win32]"